# 01 — Dataset exploration

This notebook inspects the local copy of Anish Kumar's Kaggle dataset
[Days-death-to-a-banana](https://www.kaggle.com/datasets/anishkumar00/days-death-to-a-banana).
It answers three questions before any model is trained:

1. What files and columns are actually here?
2. What does `days_left` measure?
3. Which splits would leak, because the same picture appears twice or because a filename already contains the answer?

The notebook only reads the extract. It does not resize images, train a model, or rewrite the CSV.

The written summary of the same run is `docs/phase1_dataset_report.md`.


## Setup

The extract stays where it was downloaded. Raw images are gitignored on purpose:
the folder mixes ordinary photos with watermarked stock images, and a clone of
this repo should re-download from Kaggle rather than inherit a private Desktop path.

Search order:

1. `BANANA_DATA_DIR`, if set.
2. `ml/data/raw/` inside this repository.
3. `~/Desktop/archive`, which is where the extract was found on 9 November 2026.

A hit is a directory that contains both `banana_days_471_synced.csv` and `banana_images_jpg/`.


In [ ]:
from __future__ import annotations

import hashlib
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

CSV_NAME = "banana_days_471_synced.csv"
IMAGE_DIR_NAME = "banana_images_jpg"
FILENAME_RE = re.compile(r"^banana_(\d+)_day_(\d+)\.jpg$")


def resolve_data_dir() -> Path:
    """Return the extracted dataset directory, or raise with the paths that were tried."""
    candidates: list[Path] = []
    override = os.environ.get("BANANA_DATA_DIR")
    if override:
        candidates.append(Path(override).expanduser())

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "ml" / "data" / "raw")

    candidates.append(Path.home() / "Desktop" / "archive")

    tried: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        tried.append(candidate)
        csv_path = candidate / CSV_NAME
        image_dir = candidate / IMAGE_DIR_NAME
        if csv_path.is_file() and image_dir.is_dir():
            return candidate

    rendered = "\n".join(f"  - {path}" for path in tried)
    raise FileNotFoundError(
        "Could not find the Kaggle extract. Looked for "
        f"{CSV_NAME} and {IMAGE_DIR_NAME} in:\n{rendered}"
    )


def parse_filename(name: str) -> tuple[int, int] | None:
    """Return (banana_number, day_token) from a basename, or None if the pattern fails."""
    match = FILENAME_RE.match(name)
    if match is None:
        return None
    return int(match.group(1)), int(match.group(2))


def expected_days_left(day_token: int) -> int:
    """Recode a filename day token the way this CSV recodes it.

    Day token 0 and day token 1 are both stored as 8. Every other token uses
    ``9 - day_token``. This describes the file that was shipped. It is not a
    claim that the number was measured with a ripeness protocol.
    """
    if day_token == 0:
        return 8
    return 9 - day_token


def file_md5(path: Path) -> str:
    """Hash the raw file bytes. Identical hashes are the same JPEG, not merely similar photos."""
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 16), b""):
            digest.update(chunk)
    return digest.hexdigest()


def difference_hash(path: Path, hash_size: int = 8) -> tuple[int, ...]:
    """Build a 64-bit difference hash.

    The image is shrunk to a 9-by-8 grayscale thumbnail. Each bit records whether
    a pixel is brighter than the pixel on its right. Hamming distance then counts
    disagreeing bits. Distance 0 is a very strong match between different files.
    Distance 1 through 4 is a candidate copy and still needs a human look.
    """
    image = Image.open(path).convert("L").resize(
        (hash_size + 1, hash_size),
        Image.Resampling.LANCZOS,
    )
    pixels = list(image.get_flattened_data())
    width = hash_size + 1
    bits: list[int] = []
    for y in range(hash_size):
        row = y * width
        for x in range(hash_size):
            bits.append(int(pixels[row + x] > pixels[row + x + 1]))
    return tuple(bits)


def hamming(left: tuple[int, ...], right: tuple[int, ...]) -> int:
    """Count bits that differ between two equal-length hashes."""
    return sum(a != b for a, b in zip(left, right, strict=True))


DATA_DIR = resolve_data_dir()
CSV_PATH = DATA_DIR / CSV_NAME
IMAGE_DIR = DATA_DIR / IMAGE_DIR_NAME
print(f"data directory: {DATA_DIR}")


## Labels

`days_left` is read as an integer. A blank or a decimal would matter, because a
regression target and a disguised class index are different problems. The first
rows show the pattern the rest of the file follows: many images, and a label
that steps down as the day token steps up.


In [ ]:
import csv

with CSV_PATH.open(newline="") as handle:
    reader = csv.DictReader(handle)
    columns = list(reader.fieldnames or [])
    raw_rows = list(reader)

print("columns:", columns)
print("csv rows:", len(raw_rows))

non_integer = [
    row["days_left"]
    for row in raw_rows
    if not re.fullmatch(r"-?\d+", row["days_left"].strip())
]
print("non-integer days_left values:", non_integer)

labels: dict[str, int] = {}
label_conflicts: dict[str, set[int]] = defaultdict(set)
row_counts: Counter[str] = Counter()
for row in raw_rows:
    name = row["image_filename"]
    value = int(row["days_left"])
    row_counts[name] += 1
    label_conflicts[name].add(value)
    labels.setdefault(name, value)

repeated = {name: count for name, count in row_counts.items() if count > 1}
conflicts = {name: sorted(values) for name, values in label_conflicts.items() if len(values) > 1}
print("distinct filenames:", len(labels))
print("filenames repeated in the CSV:", repeated)
print("filenames with conflicting days_left:", conflicts)
print("first 8 rows:")
for row in raw_rows[:8]:
    print(f"  {row['image_filename']:>24}  days_left={row['days_left']}")


## Joining labels to files

The only join key is the basename. A row whose file is missing cannot be trained
on. A file with no row has no supervised target. Both lists should be empty
before anyone starts a dataloader, or the leftovers need an explicit decision.


In [ ]:
image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
image_names = [path.name for path in image_paths]
image_set = set(image_names)
label_set = set(labels)

print("jpeg files:", len(image_names))
print("in CSV, missing on disk:", sorted(label_set - image_set))
print("on disk, missing from CSV:", sorted(image_set - label_set))

unparsed = [name for name in sorted(label_set | image_set) if parse_filename(name) is None]
print("names that break banana_{id}_day_{day}.jpg:", unparsed)


## What `days_left` is

Every labeled filename is parsed into a banana number and a day token. The check
below asks whether `days_left` adds information beyond that token.

If the two columns always agree, the CSV is a renamed copy of the filename.
A model, a dataloader, or a filename feature that can see `day_08` can then
emit `1` without looking at the photograph. That result would be leakage, not
shelf-life estimation.


In [ ]:
mismatches: list[tuple[str, int, int, int]] = []
for name, value in sorted(labels.items()):
    parsed = parse_filename(name)
    assert parsed is not None
    _banana_number, day_token = parsed
    expected = expected_days_left(day_token)
    if value != expected:
        mismatches.append((name, day_token, value, expected))

print("labeled files whose days_left breaks the filename rule:", len(mismatches))
for item in mismatches:
    print(" ", item)

by_token: dict[int, Counter[int]] = defaultdict(Counter)
for name, value in labels.items():
    parsed = parse_filename(name)
    assert parsed is not None
    by_token[parsed[1]][value] += 1

print("day token -> days_left counts")
for day_token in sorted(by_token):
    print(f"  day {day_token:02d}: {dict(sorted(by_token[day_token].items()))}")


## How many images sit on each value

Counts use distinct filenames. A repeated CSV row is one image, not two.
The plot is the whole modeling consequence of this table: the values are uneven,
and `days_left = 7` is a thin bin. A later test split has to keep that in view,
because a handful of those images can move MAE by a lot.

The day token is on the chart only as the filename's other spelling of the same
number. It is not a second measurement.


In [ ]:
distribution = Counter(labels.values())
print("distinct labeled images by days_left:")
for value in sorted(distribution):
    print(f"  {value}: {distribution[value]}")

csv_distribution = Counter(int(row["days_left"]) for row in raw_rows)
print("CSV rows by days_left, repeats included:")
for value in sorted(csv_distribution):
    print(f"  {value}: {csv_distribution[value]}")

fig, ax = plt.subplots(figsize=(7, 3.5))
values = sorted(distribution)
counts = [distribution[value] for value in values]
ax.bar([str(value) for value in values], counts, color="#c4a35a")
ax.set_xlabel("days_left")
ax.set_ylabel("labeled images")
ax.set_title("Label counts are uneven, and 7 is the thin bin")
for x, count in enumerate(counts):
    ax.text(x, count + 1, str(count), ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, max(counts) + 12)
fig.tight_layout()
plt.show()


## Image properties

Resolution and file size are part of the data, not just loading details.
If one label is mostly tiny studio crops and another is mostly phone photos,
a network can score well by detecting the camera rather than the fruit.

EXIF would have been the clean way to test the "same banana, photographed each
day" story, because a capture timestamp settles it. This folder has none.


In [ ]:
sizes: list[tuple[int, int]] = []
modes: Counter[str] = Counter()
exif_count = 0
byte_sizes: list[int] = []
long_side_by_label: dict[int, list[int]] = defaultdict(list)

for path in image_paths:
    byte_sizes.append(path.stat().st_size)
    with Image.open(path) as image:
        sizes.append(image.size)
        modes[image.mode] += 1
        if image.getexif():
            exif_count += 1
        if path.name in labels:
            long_side_by_label[labels[path.name]].append(max(image.size))

print("modes:", dict(modes))
print("images with EXIF:", exif_count, "of", len(image_paths))
print("distinct resolutions:", len(set(sizes)))
print("most common resolutions:")
for (width, height), count in Counter(sizes).most_common(8):
    print(f"  {width}x{height}: {count}")

byte_array = np.array(byte_sizes)
print(
    "file size bytes:"
    f" min={byte_array.min()} median={int(np.median(byte_array))}"
    f" max={byte_array.max()} under_5kb={int((byte_array < 5000).sum())}"
)

print("share of labeled images with long side < 500 px:")
for value in sorted(long_side_by_label):
    sides = long_side_by_label[value]
    small = sum(side < 500 for side in sides)
    print(f"  days_left {value}: {small}/{len(sides)}")


## Exact copies

MD5 compares bytes. Two names with one hash are the same file saved twice,
whatever their banana numbers say. If those names also carry different
`days_left` values, the photograph has no single target. Keeping both rows
teaches the loss two answers for one input, and a random split lets the model
memorize the training copy.


In [ ]:
hash_to_names: dict[str, list[str]] = defaultdict(list)
for path in image_paths:
    hash_to_names[file_md5(path)].append(path.name)

duplicate_groups = {digest: names for digest, names in hash_to_names.items() if len(names) > 1}
conflict_groups = []
same_label_groups = []
for names in duplicate_groups.values():
    present = {labels[name] for name in names if name in labels}
    if len(present) > 1:
        conflict_groups.append(sorted(names))
    else:
        same_label_groups.append(sorted(names))

files_in_groups = sum(len(names) for names in duplicate_groups.values())
print("distinct file hashes:", len(hash_to_names))
print("hashes shared by multiple names:", len(duplicate_groups))
print("files involved in those hashes:", files_in_groups)
print("groups with one days_left:", len(same_label_groups))
print("groups with conflicting days_left:", len(conflict_groups))
print("conflicting groups:")
for names in sorted(conflict_groups, key=len, reverse=True):
    rendered = ", ".join(f"{name}:{labels.get(name, 'unlabeled')}" for name in names)
    print(f"  {rendered}")


## Near copies

Some copies were opened and saved again, so the bytes differ and MD5 misses them.
The difference hash is a screen for those. Pairs at distance 0 with different
bytes are resized-identical. Pairs out to distance 4 are worth opening by hand.
This threshold is a review rule, not a biological definition of "same banana."

Pairs already caught by MD5 are skipped here. Those were listed above.


In [ ]:
NEAR_DISTANCE = 4
hashes = {path.name: difference_hash(path) for path in image_paths}
md5 = {name: digest for digest, names in hash_to_names.items() for name in names}

near_pairs: list[tuple[int, str, str]] = []
names = [path.name for path in image_paths]
for i, left in enumerate(names):
    for right in names[i + 1 :]:
        if md5[left] == md5[right]:
            continue
        distance = hamming(hashes[left], hashes[right])
        if distance <= NEAR_DISTANCE:
            near_pairs.append((distance, left, right))

near_pairs.sort()
print(f"different-byte pairs with difference-hash distance <= {NEAR_DISTANCE}:", len(near_pairs))
for distance, left, right in near_pairs:
    print(
        f"  distance {distance}: {left} (days_left={labels.get(left, 'unlabeled')})"
        f" ~ {right} (days_left={labels.get(right, 'unlabeled')})"
    )


## Does the banana number track one fruit?

A number that appears on day 1 and day 8 is a candidate specimen id. Two checks
reject that reading for this folder.

The coverage table shows the panel is full of holes: most numbers do not have
all eight days, and a block of numbers exists only on day 8.

The distance table is the direct photo check. If the number meant "this fruit,
tomorrow," photos that share a number would sit much closer together than two
random photos. The medians below are the comparison. The random sample uses
seed 0 so the figure stays fixed across reruns.


In [ ]:
by_id: dict[int, set[int]] = defaultdict(set)
for name in labels:
    parsed = parse_filename(name)
    assert parsed is not None
    banana_number, day_token = parsed
    by_id[banana_number].add(day_token)

full_week = {1, 2, 3, 4, 5, 6, 7, 8}
n_days = Counter(len(days) for days in by_id.values())
only_day_8 = sorted(number for number, days in by_id.items() if days == {8})
print("distinct banana numbers:", len(by_id))
print("numbers present on day tokens 1 through 8:", sum(days == full_week for days in by_id.values()))
print("numbers present on day 8 only:", len(only_day_8), only_day_8)
print("numbers by how many distinct day tokens they have:")
for n_present, count in sorted(n_days.items()):
    print(f"  {n_present} day tokens: {count} numbers")

within_distances: list[int] = []
id_to_names: dict[int, list[str]] = defaultdict(list)
for name in labels:
    parsed = parse_filename(name)
    assert parsed is not None
    id_to_names[parsed[0]].append(name)

for names_for_id in id_to_names.values():
    for i, left in enumerate(names_for_id):
        for right in names_for_id[i + 1 :]:
            within_distances.append(hamming(hashes[left], hashes[right]))

rng = np.random.default_rng(0)
labeled_names = np.array(sorted(labels))
random_distances: list[int] = []
for _ in within_distances:
    left, right = rng.choice(labeled_names, size=2, replace=False)
    random_distances.append(hamming(hashes[str(left)], hashes[str(right)]))


def summarize(distances: list[int]) -> str:
    """Format min, quartiles, and mean for a list of Hamming distances."""
    array = np.array(distances)
    quartiles = np.percentile(array, [0, 25, 50, 75, 100])
    close = int((array <= NEAR_DISTANCE).sum())
    return (
        f"n={len(array)} min={quartiles[0]:.0f} p25={quartiles[1]:.0f} "
        f"median={quartiles[2]:.0f} p75={quartiles[3]:.0f} max={quartiles[4]:.0f} "
        f"mean={array.mean():.1f} within_{NEAR_DISTANCE}bits={close}"
    )


adjacent_distances: list[int] = []
for names_for_id in id_to_names.values():
    ordered_days: list[tuple[int, str]] = []
    for name in names_for_id:
        parsed = parse_filename(name)
        assert parsed is not None
        ordered_days.append((parsed[1], name))
    ordered_days.sort()
    for (left_day, left), (right_day, right) in zip(ordered_days, ordered_days[1:]):
        if right_day == left_day + 1:
            adjacent_distances.append(hamming(hashes[left], hashes[right]))

print("same banana number:", summarize(within_distances))
print("adjacent day tokens, same number:", summarize(adjacent_distances))
print("random labeled pairs:", summarize(random_distances))


## Pictures worth opening

Same banana number, three day tokens: these should be one fruit aging if the
filename is a time series. They are three different shoots.

The middle row is one byte-identical photograph stored under two names and two
labels. The last row is the malformed day token, a phone-sized frame, and a
high id that exists only at the end of the index.

Read the caption, then the fruit. The caption's number came from the filename.


In [ ]:
examples: list[tuple[str, str]] = [
    ("banana_01_day_01.jpg", "number 01, day 01, days_left 8"),
    ("banana_01_day_04.jpg", "number 01, day 04, days_left 5"),
    ("banana_01_day_08.jpg", "number 01, day 08, days_left 1"),
    ("banana_36_day_06.jpg", "same bytes as the next frame, days_left 3"),
    ("banana_54_day_08.jpg", "same bytes as the previous frame, days_left 1"),
    ("banana_10_day_08.jpg", "number 10, day 08, days_left 1"),
    ("banana_33_day_0.jpg", "day token 0, days_left 8"),
    ("banana_14_day_08.jpg", "960x1280 phone frame, days_left 1"),
    ("banana_99_day_08.jpg", "number 99 exists only on day 08"),
]

fig, axes = plt.subplots(3, 3, figsize=(10, 8))
for ax, (name, caption) in zip(axes.ravel(), examples, strict=True):
    with Image.open(IMAGE_DIR / name) as image:
        ax.imshow(image)
    ax.set_title(caption, fontsize=8)
    ax.axis("off")
fig.suptitle("The filename number is not one banana aging in place", fontsize=12)
fig.tight_layout()
plt.show()


## Mean color, as a weak check

Average RGB of a 64×64 thumbnail mixes the fruit with the table, the wall, and
the black studio backdrop. It is here to see whether the label even moves with
overall color. A smooth slide from green toward brown would support a ripeness
scale. A jumpy average means the label is not a simple color reading, which
matches the mixed scenes above.

Duplicate filenames are included once. Two different names that share bytes are
both included, under their own labels.


In [ ]:
rgb_by_label: dict[int, list[tuple[float, float, float]]] = defaultdict(list)
for path in image_paths:
    if path.name not in labels:
        continue
    with Image.open(path) as image:
        thumb = np.asarray(image.convert("RGB").resize((64, 64)), dtype=np.float64)
    rgb_by_label[labels[path.name]].append(tuple(thumb.mean(axis=(0, 1))))

print("mean RGB by days_left:")
for value in sorted(rgb_by_label):
    means = np.array(rgb_by_label[value])
    red, green, blue = means.mean(axis=0)
    print(f"  {value}: n={len(means)} R={red:.1f} G={green:.1f} B={blue:.1f}")


## Cleaning count, not a cleaned dataset

This cell only counts. Proposed rule, for a later decision:

- leave the three unlabeled files out of supervised training
- drop a whole hash group when its copies disagree on `days_left`
- keep one file when every copy of a hash agrees

Near-duplicate pairs are printed above and are not removed by this count.
Several of them cross labels, so exact-hash cleaning is incomplete. No file is
renamed, deleted, or copied by this notebook.


In [ ]:
unlabeled = sorted(image_set - label_set)
conflict_hashes = {
    digest
    for digest, names in hash_to_names.items()
    if len({labels[name] for name in names if name in labels}) > 1
}
unambiguous = [
    digest
    for digest, names in hash_to_names.items()
    if digest not in conflict_hashes and any(name in labels for name in names)
]
print("unlabeled files held out:", unlabeled)
print("hashes with conflicting labels:", len(conflict_hashes))
print("hashes with one consistent label:", len(unambiguous))
print("near-duplicate pairs still to review:", len(near_pairs))


## Decisions this notebook is not making

The filename must not become a feature. The banana number must not become the
split key. A credible split groups exact copies and reviewed near-copies so
that one photograph cannot land in both train and test.

Whether to train on this extract at all is a product decision. The label is the
day token rewritten as `days_left`. Greener studio photos and browner phone
photos both occur, and so do unrelated scenes that share a number. A low error
on this CSV is evidence about the day index, not a measured count of edible days.

Phase 2 waits on that decision. The next code worth writing is a tested parser
for the filename and an audit of the hash groups. Training comes after the
split exists.
